# 07 — Graph Tools
Test each deterministic tool in `GraphTools` against a live Neo4j instance.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.tools.graph_tools import GraphTools

settings = get_neo4j_settings()
repo = Neo4jRepository(**vars(settings))
tools = GraphTools(repo)
repo.test_connection()
print("Connected")

Connected


## Helper — pretty-print a ToolResult

In [3]:
def show(result):
    status = "✓" if result.success else "✗"
    print(f"[{status}] {result.tool_name}  ({result.duration_ms} ms)")
    print(f"    INPUT  : {result.input}")
    print(f"    SUMMARY: {result.summary}")
    if not result.success:
        print(f"    ERROR  : {result.error}")

In [4]:
COMPANY = "TELEFONICA O2 HOLDINGS LIMITED"

## entity_lookup

In [5]:
r = tools.entity_lookup("telefonica")
show(r)
if r.data:
    display(pd.DataFrame(r.data))

[✓] entity_lookup  (23.0 ms)
    INPUT  : {'name': 'telefonica'}
    SUMMARY: Found 10 company match(es) for 'telefonica'. Top match: 'TELEFONICA UK LIMITED' (number: 01743099, status: Active).


,name,company_number,status,country_of_origin,score
0,TELEFONICA UK LIMITED,01743099,Active,United Kingdom,6.352769
1,TELEFONICA DIGITAL LIMITED,07884976,Active,United Kingdom,6.352769
2,Telefonica S A,A 28015865,None,None,6.352769
3,TELEFONICA UK HOLDINGS LIMITED,03722259,Active,United Kingdom,5.653621
4,TELEFONICA O2 UK LIMITED,04330394,Active,United Kingdom,5.653621
5,TELEFONICA O2 HOLDINGS LIMITED,05310128,Active,United Kingdom,5.653621
6,TELEFONICA GERMANY HOLDINGS LIMITED,08202391,Active,United Kingdom,5.653621
7,TELEFONICA INSURANCE UK BRANCH,FC029774,Active,LUXEMBOURG,5.653621
8,TELEFONICA UK PENSION TRUSTEE LIMITED,04267552,Active,United Kingdom,5.093103
9,Telefonica Tech Uk & Ireland Limited,11243186,None,None,5.093103


## company_profile

In [6]:
r = tools.company_profile(COMPANY)
show(r)
if r.data:
    print("\nCompany:")
    display(pd.DataFrame([r.data["company"]]))
    print("\nAddress:")
    display(pd.DataFrame([r.data["address"]]) if r.data["address"] else "none")
    print("\nSIC codes:")
    display(pd.DataFrame(r.data["sic_codes"]) if r.data["sic_codes"] else "none")
    print("\nDirect owners:")
    display(pd.DataFrame(r.data["direct_owners"]) if r.data["direct_owners"] else "none")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `locality` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=19, offset=222>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 222, 'line': 6, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (c:Company {name: $name})-[:REGISTERED_AT]->(a:Address)\n            RETURN\n                a.address_line_1    AS address_line_1,\n                a.address_line_2    AS address_line_2,\n                a.locality          AS locality,\n                a.region            AS region,\n     

[✗] company_profile  (85.7 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED'}
    SUMMARY: company_profile failed: sequence item 0: expected str instance, NoneType found
    ERROR  : sequence item 0: expected str instance, NoneType found


## expand_ownership

In [7]:
r = tools.expand_ownership(COMPANY, max_depth=5)
show(r)
if r.data:
    print("\nOwnership paths:")
    display(pd.DataFrame(r.data["paths"]) if r.data["paths"] else "none")
    print("\nUltimate individual owners:")
    display(pd.DataFrame(r.data["ultimate_owners"]) if r.data["ultimate_owners"] else "none")

[✓] expand_ownership  (46.4 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED', 'max_depth': 5}
    SUMMARY: Found 1 ownership hop(s) across depths [1] for 'TELEFONICA O2 HOLDINGS LIMITED'. All chains terminate at corporate entities — no individual UBOs found.

Ownership paths:


,path_depth,hop,from_name,from_labels,ownership_pct_min,ownership_pct_max,ownership_controls,to_name,to_labels
0,1,1,Telefonica S A,[Company],75,100,[ownership-of-shares-75-to-100-percent],TELEFONICA O2 HOLDINGS LIMITED,[Company]



Ultimate individual owners:


'none'

## shared_address_check

In [8]:
r = tools.shared_address_check(COMPANY)
show(r)
if r.data and r.data["co_located_companies"]:
    df = pd.DataFrame(r.data["co_located_companies"])
    print(f"\nCo-located companies (showing first 10 of {r.data['total_co_located']}):")
    display(df.head(10))

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `locality` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=19, offset=222>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 222, 'line': 6, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (c:Company {name: $name})-[:REGISTERED_AT]->(a:Address)\n            RETURN\n                a.address_line_1    AS address_line_1,\n                a.address_line_2    AS address_line_2,\n                a.locality          AS locality,\n                a.region            AS region,\n     

[✓] shared_address_check  (46.0 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED'}
    SUMMARY: 'TELEFONICA O2 HOLDINGS LIMITED' shares its address (None) with 131 other companies (126 active). Address co-location risk: HIGH.

Co-located companies (showing first 10 of 131):


,company_name,company_number,status,postal_code,address_line_1
0,00084968 LIMITED,00084968,Active,None,HIGHDOWN HOUSE
1,15 SPITAL SQUARE RESIDENTS ASSOCIATION LTD,02197789,Active,None,HIGHDOWN HOUSE
2,"AKARI THERAPEUTICS, PLC",05252842,Active,None,HIGHDOWN HOUSE
3,ALPHAWAVE IP GROUP PLC,13073661,Active,None,HIGHDOWN HOUSE
4,ARCUS INVESTMENT HOLDINGS LIMITED,12974482,Active,None,HIGHDOWN HOUSE
5,ARCUS INVESTMENT LIMITED,03582673,Active,None,HIGHDOWN HOUSE
6,ASA INTERNATIONAL GROUP PLC,11361159,Active,None,HIGHDOWN HOUSE
7,BALTIC CLASSIFIEDS GROUP PLC,13357598,Active,None,HIGHDOWN HOUSE
8,BENCHMARK ANIMAL HEALTH GROUP LIMITED,07330728,Active,None,HIGHDOWN HOUSE
9,BENCHMARK ANIMAL HEALTH LIMITED,08872045,Active,None,HIGHDOWN HOUSE


## sic_context

In [9]:
r = tools.sic_context(COMPANY)
show(r)
if r.data:
    print("\nSIC codes:")
    display(pd.DataFrame(r.data["sic_codes"]) if r.data["sic_codes"] else "none")
    print("\nPeers (first 10):")
    display(pd.DataFrame(r.data["peers"]).head(10) if r.data["peers"] else "none")

[✓] sic_context  (252.2 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED'}
    SUMMARY: 'TELEFONICA O2 HOLDINGS LIMITED' operates under SIC code(s): None (Other telecommunications activities), None (Non-trading company). Found 50 peer company(ies) sharing at least one code.

SIC codes:


,sic_code,sic_description
0,None,Other telecommunications activities
1,None,Non-trading company



Peers (first 10):


,company_name,company_number,status,shared_sic_codes,shared_sic_descriptions
0,"""""""BEECHBANK COURT"""" MANAGEMENT COMPANY LIMITED""",01382560,Active,[],[Non-trading company]
1,"""""""CANNON CHAMBERS"""" 17 CANNON PLACE MANAGEMEN...",02076454,Active,[],[Non-trading company]
2,"""""""INTOUCH COMMUNICATION SERVICES"""" LIMITED""",03606467,Active,[],[Other telecommunications activities]
3,"""""""ST MARY'S HEATH"""" AT ARMTHORPE PROPERTY MAN...",05424123,Active,[],[Non-trading company]
4,"""CHADWELL HEATH """"A"""" FLAT MANAGEMENT LIMITED""",02184970,Active,[],[Non-trading company]
5,"""CHADWELL HEATH """"B"""" FLAT MANAGEMENT LIMITED""",02185149,Active,[],[Non-trading company]
6,"""CHADWELL HEATH """"B"""" MANAGEMENT LIMITED""",01823617,Active,[],[Non-trading company]
7,"""CHADWELL HEATH """"C"""" MANAGEMENT LIMITED""",01823618,Active,[],[Non-trading company]
8,"""CHADWELL HEATH """"D"""" MANAGEMENT LIMITED""",01941575,Active,[],[Non-trading company]
9,"""CHADWELL HEATH """"E"""" MANAGEMENT LIMITED""",01950602,Active,[],[Non-trading company]


## Error handling — unknown company

In [10]:
r = tools.company_profile("THIS COMPANY DOES NOT EXIST LTD")
show(r)  # success=True, data=None, summary explains the miss

[✓] company_profile  (18.6 ms)
    INPUT  : {'company_name': 'THIS COMPANY DOES NOT EXIST LTD'}
    SUMMARY: No company found with exact name 'THIS COMPANY DOES NOT EXIST LTD'.


## Close

In [11]:
repo.close()
print("Driver closed")

Driver closed
